In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from itertools import combinations

def create_advanced_features(data):
    """
    Create advanced features through systematic combinations of existing features
    """
    df = data.copy()
    
    # ========== AGGREGATE FEATURES ==========
    df['M*'] = df[[f'M{i}' for i in range(1, 19)]].sum(axis=1)
    df['E*'] = df[[f'E{i}' for i in range(1, 21)]].sum(axis=1)
    df['I*'] = df[[f'I{i}' for i in range(1, 10)]].sum(axis=1)
    df['P*'] = df[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1)
    df['V*'] = df[[f'V{i}' for i in range(1, 14)]].sum(axis=1)
    df['S*'] = df[[f'S{i}' for i in range(1, 13)]].sum(axis=1)
    df['D*'] = df[[f'D{i}' for i in range(1, 10)]].sum(axis=1)
    
    # Core feature groups
    agg_features = ['M*', 'E*', 'I*', 'P*', 'V*', 'S*', 'D*']
    individual_features = ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9', 'E16', 'E17', 'E19', 'E20',
                          'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S12', 'I2', 'P8', 'P9', 'P10']
    
    # ========== 1. TWO-WAY RATIO COMBINATIONS ==========
    # All pairwise ratios between aggregate features
    for feat1, feat2 in combinations(agg_features, 2):
        df[f'{feat1}_{feat2}_ratio'] = df[feat1] / (df[feat2] + 1e-6)
        df[f'{feat2}_{feat1}_ratio'] = df[feat2] / (df[feat1] + 1e-6)
    
    # ========== 2. TWO-WAY PRODUCT COMBINATIONS ==========
    # All pairwise products
    for feat1, feat2 in combinations(agg_features, 2):
        df[f'{feat1}_x_{feat2}'] = df[feat1] * df[feat2]
    
    # ========== 3. TWO-WAY DIFFERENCE COMBINATIONS ==========
    for feat1, feat2 in combinations(agg_features, 2):
        df[f'{feat1}_minus_{feat2}'] = df[feat1] - df[feat2]
    
    # ========== 4. TWO-WAY SUM COMBINATIONS ==========
    for feat1, feat2 in combinations(agg_features, 2):
        df[f'{feat1}_plus_{feat2}'] = df[feat1] + df[feat2]
    
    # ========== 5. THREE-WAY COMBINATIONS ==========
    # Important 3-feature combinations
    three_way_groups = [
        ('M*', 'E*', 'V*'),
        ('M*', 'P*', 'V*'),
        ('M*', 'S*', 'V*'),
        ('E*', 'I*', 'P*'),
        ('V*', 'S*', 'D*'),
        ('M*', 'E*', 'S*'),
        ('P*', 'I*', 'V*'),
        ('M*', 'P*', 'I*'),
        ('E*', 'V*', 'S*'),
        ('M*', 'I*', 'S*')
    ]
    
    for f1, f2, f3 in three_way_groups:
        # Product combinations
        df[f'{f1}_x_{f2}_x_{f3}'] = df[f1] * df[f2] * df[f3]
        
        # Ratio combinations
        df[f'{f1}_x_{f2}_div_{f3}'] = (df[f1] * df[f2]) / (df[f3] + 1e-6)
        df[f'{f1}_div_{f2}_x_{f3}'] = df[f1] / ((df[f2] * df[f3]) + 1e-6)
        
        # Sum then divide
        df[f'{f1}_plus_{f2}_div_{f3}'] = (df[f1] + df[f2]) / (df[f3] + 1e-6)
        
        # Weighted combinations
        df[f'{f1}_weighted_{f2}_{f3}'] = 0.5*df[f1] + 0.3*df[f2] + 0.2*df[f3]
    
    # ========== 6. FOUR-WAY COMBINATIONS ==========
    four_way_groups = [
        ('M*', 'E*', 'P*', 'V*'),
        ('M*', 'E*', 'I*', 'S*'),
        ('M*', 'V*', 'S*', 'D*'),
        ('E*', 'I*', 'P*', 'V*'),
        ('M*', 'P*', 'S*', 'V*')
    ]
    
    for f1, f2, f3, f4 in four_way_groups:
        # Complex ratios
        df[f'{f1}_x_{f2}_div_{f3}_x_{f4}'] = (df[f1] * df[f2]) / ((df[f3] * df[f4]) + 1e-6)
        df[f'{f1}_plus_{f2}_div_{f3}_plus_{f4}'] = (df[f1] + df[f2]) / ((df[f3] + df[f4]) + 1e-6)
        
        # Weighted 4-way
        df[f'{f1}_{f2}_{f3}_{f4}_weighted'] = 0.4*df[f1] + 0.3*df[f2] + 0.2*df[f3] + 0.1*df[f4]
    
    # ========== 7. INDIVIDUAL FEATURE COMBINATIONS WITH AGGREGATES ==========
    # Combine important individual features with aggregates
    for ind_feat in ['E7', 'E8', 'S4', 'P8', 'P9']:
        for agg_feat in agg_features:
            df[f'{ind_feat}_x_{agg_feat}'] = df[ind_feat] * df[agg_feat]
            df[f'{ind_feat}_div_{agg_feat}'] = df[ind_feat] / (df[agg_feat] + 1e-6)
            df[f'{agg_feat}_div_{ind_feat}'] = df[agg_feat] / (df[ind_feat] + 1e-6)
    
    # ========== 8. VOLATILITY-NORMALIZED COMBINATIONS ==========
    base_features = ['M*', 'E*', 'P*', 'I*', 'S*']
    for feat in base_features:
        df[f'{feat}_vol_norm'] = df[feat] / (df['V*'] + 1e-6)
        df[f'{feat}_vol_adj_squared'] = (df[feat] ** 2) / (df['V*'] + 1e-6)
    
    # ========== 9. RISK-ADJUSTED MULTI-COMBINATIONS ==========
    # Risk = V* + D* (volatility + dummy factors)
    df['risk_composite'] = df['V*'] + df['D*']
    
    risk_adj_combos = [
        ('M*', 'E*'),
        ('M*', 'P*'),
        ('E*', 'P*'),
        ('M*', 'S*'),
        ('E*', 'I*'),
        ('P*', 'I*')
    ]
    
    for f1, f2 in risk_adj_combos:
        df[f'{f1}_x_{f2}_risk_adj'] = (df[f1] * df[f2]) / (df['risk_composite'] + 1e-6)
        df[f'{f1}_plus_{f2}_risk_adj'] = (df[f1] + df[f2]) / (df['risk_composite'] + 1e-6)
    
    # ========== 10. MOMENTUM COMBINATIONS ==========
    # Momentum = Market * Sentiment interactions
    momentum_pairs = [
        ('M*', 'E*'),
        ('M*', 'S*'),
        ('M*', 'P*'),
        ('E*', 'S*')
    ]
    
    for f1, f2 in momentum_pairs:
        df[f'{f1}_{f2}_momentum'] = df[f1] * df[f2]
        df[f'{f1}_{f2}_momentum_vol_adj'] = (df[f1] * df[f2]) / (df['V*'] + 1e-6)
    
    # ========== 11. SHARPE-TYPE COMBINATIONS ==========
    # (Return proxy - Risk proxy) / Volatility
    sharpe_combos = [
        ('M*', 'E*', 'V*'),
        ('M*', 'P*', 'V*'),
        ('E*', 'I*', 'V*'),
        ('M*', 'S*', 'V*')
    ]
    
    for f1, f2, f3 in sharpe_combos:
        df[f'{f1}_minus_{f2}_div_{f3}_sharpe'] = (df[f1] - df[f2]) / (df[f3] + 1e-6)
    
    # ========== 12. POLYNOMIAL COMBINATIONS ==========
    for feat in agg_features:
        df[f'{feat}_squared'] = df[feat] ** 2
        df[f'{feat}_cubed'] = df[feat] ** 3
        df[f'{feat}_sqrt'] = np.sqrt(np.abs(df[feat]))
    
    # ========== 13. EXPONENTIAL COMBINATIONS ==========
    for feat in ['M*', 'V*', 'S*', 'E*']:
        # Normalize before exp to avoid overflow
        norm_val = df[feat] / (df[feat].abs().max() + 1e-6)
        df[f'{feat}_exp'] = np.exp(norm_val)
        df[f'{feat}_log'] = np.log1p(np.abs(df[feat]))
    
    # ========== 14. CROSS-CATEGORY COMBINATIONS ==========
    # Market + Economic vs Risk
    df['ME_combined'] = df['M*'] + df['E*']
    df['ME_vs_risk'] = (df['M*'] + df['E*']) / (df['V*'] + df['D*'] + 1e-6)
    
    # Price + Interest vs Market
    df['PI_combined'] = df['P*'] + df['I*']
    df['PI_vs_market'] = (df['P*'] + df['I*']) / (df['M*'] + 1e-6)
    
    # Sentiment + Dummy vs Volatility
    df['SD_combined'] = df['S*'] + df['D*']
    df['SD_vs_vol'] = (df['S*'] + df['D*']) / (df['V*'] + 1e-6)
    
    # ========== 15. RATIO OF RATIOS ==========
    df['ME_to_PV_ratio'] = (df['M*'] / (df['E*'] + 1e-6)) / (df['P*'] / (df['V*'] + 1e-6) + 1e-6)
    df['MI_to_PV_ratio'] = (df['M*'] / (df['I*'] + 1e-6)) / (df['P*'] / (df['V*'] + 1e-6) + 1e-6)
    df['ES_to_IV_ratio'] = (df['E*'] / (df['S*'] + 1e-6)) / (df['I*'] / (df['V*'] + 1e-6) + 1e-6)
    
    # ========== 16. NORMALIZED SPREADS ==========
    df['ME_spread_norm'] = (df['M*'] - df['E*']) / (df['M*'] + df['E*'] + 1e-6)
    df['PV_spread_norm'] = (df['P*'] - df['V*']) / (df['P*'] + df['V*'] + 1e-6)
    df['IS_spread_norm'] = (df['I*'] - df['S*']) / (df['I*'] + df['S*'] + 1e-6)
    
    # ========== 17. INDIVIDUAL FEATURE PAIRS ==========
    individual_pairs = [
        ('E7', 'E8'),
        ('S4', 'S6'),
        ('P8', 'P9'),
        ('E7', 'S4'),
        ('E8', 'P8')
    ]
    
    for f1, f2 in individual_pairs:
        df[f'{f1}_x_{f2}'] = df[f1] * df[f2]
        df[f'{f1}_div_{f2}'] = df[f1] / (df[f2] + 1e-6)
        df[f'{f1}_plus_{f2}'] = df[f1] + df[f2]
        df[f'{f1}_minus_{f2}'] = df[f1] - df[f2]
    
    # ========== 18. COMPOSITE SCORES ==========
    # Market quality score
    df['market_quality'] = 0.3*df['M*'] + 0.3*df['E*'] + 0.2*df['S*'] - 0.2*df['V*']
    
    # Value score
    df['value_score'] = 0.4*df['E*'] + 0.3*df['P*'] + 0.2*df['I*'] - 0.1*df['V*']
    
    # Risk score
    df['risk_score'] = 0.5*df['V*'] + 0.3*df['D*'] + 0.2*(1/((df['S*']) + 1e-6))
    
    # Momentum score
    df['momentum_score'] = 0.5*df['M*'] + 0.3*df['S*'] - 0.2*df['D*']
    
    return df


def preprocessing(data, typ, scaler=None, imputer=None):
    """
    Enhanced preprocessing with comprehensive feature combinations
    """
    # Drop rows with missing values
    data = data.dropna()
    
    # Create all advanced features
    data = create_advanced_features(data)
    
    # Core features to always include
    core_features = ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9', 'E16', 'E17', 'E19', 'E20', 
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S12', "I2", "P8", "P9", "P10"]
    
    # Aggregated features
    agg_features = ["M*", "E*", "I*", "P*", "V*", "S*", "D*"]
    
    # All engineered features
    engineered_cols = [col for col in data.columns if col not in core_features + agg_features + ['forward_returns', 'date_id']]
    
    # Combine all features
    all_features = core_features + agg_features + engineered_cols
    
    if typ == "train":
        data = data[all_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[all_features].copy()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    # Imputation
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    # Scaling
    if typ == "train":
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data


# Usage example:
if __name__ == "__main__":
    # Load data
    train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
    
    # Process training data
    train_processed, scaler, imputer = preprocessing(train, "train")
    
    # Split features and target
    train_x = train_processed.drop(columns=["target"])
    train_y = train_processed['target']
    
    print(f"Original features: {len(train.columns)}")
    print(f"Engineered features: {len(train_x.columns)}")
    print(f"Total new features created: {len(train_x.columns) - len(train.columns) + 1}")
    print(f"\nTraining data shape: {train_x.shape}")
    print(f"Target shape: {train_y.shape}")
    
    # Show some example feature names
    print(f"\nExample combined features:")
    combo_features = [col for col in train_x.columns if '_x_' in col or '_div_' in col or '_plus_' in col][:20]
    for feat in combo_features:
        print(f"  - {feat}")

Original features: 98
Engineered features: 478
Total new features created: 381

Training data shape: (2021, 478)
Target shape: (2021,)

Example combined features:
  - M*_x_E*
  - M*_x_I*
  - M*_x_P*
  - M*_x_V*
  - M*_x_S*
  - M*_x_D*
  - E*_x_I*
  - E*_x_P*
  - E*_x_V*
  - E*_x_S*
  - E*_x_D*
  - I*_x_P*
  - I*_x_V*
  - I*_x_S*
  - I*_x_D*
  - P*_x_V*
  - P*_x_S*
  - P*_x_D*
  - V*_x_S*
  - V*_x_D*


In [3]:
train_processed

,E1,E2,E3,E4,E5,E7,E8,E9,E16,E17,...,E7_minus_S4,E8_x_P8,E8_div_P8,E8_plus_P8,E8_minus_P8,market_quality,value_score,risk_score,momentum_score,target
6969,0.041781,0.727190,0.649542,0.021277,0.979815,0.894408,0.010199,0.018862,0.780305,0.733879,...,0.894436,0.883548,0.249178,0.087579,0.009038,0.548433,0.658998,0.970678,0.214615,0.001145
6970,0.041500,0.728277,0.650503,0.019640,0.980146,0.894352,0.010233,0.018531,0.780123,0.733868,...,0.888975,0.883625,0.249179,0.087598,0.009081,0.566623,0.693179,0.971059,0.217197,0.004738
6971,0.041219,0.736669,0.657605,0.018003,0.980477,0.894295,0.010266,0.018200,0.779941,0.733856,...,0.891678,0.883479,0.249189,0.087783,0.008972,0.553952,0.705358,0.971356,0.179107,0.006016
6972,0.040938,0.747184,0.666562,0.016367,0.980807,0.894394,0.011356,0.017869,0.776877,0.893087,...,0.895608,0.885208,0.249270,0.089012,0.009859,0.556317,0.710626,0.971404,0.120261,0.001414
6973,0.040657,0.750377,0.669411,0.014730,0.981138,0.888036,0.011387,0.017538,0.776701,0.892525,...,0.874007,0.885256,0.249272,0.089048,0.009883,0.562347,0.700347,0.971149,0.153500,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.654241,0.781596,0.152209,0.331238,0.851819,0.042188,0.914626,0.627247,0.437709,...,0.855225,0.940346,0.251297,0.115256,0.042731,0.595879,0.651397,0.970267,0.267551,0.002457
8986,0.281430,0.656796,0.783496,0.150573,0.330907,0.851816,0.042187,0.914957,0.627242,0.437767,...,0.835089,0.940298,0.251299,0.115432,0.042567,0.632234,0.646550,0.970149,0.351349,0.002312
8987,0.280770,0.660234,0.786076,0.148936,0.330576,0.851813,0.042186,0.915288,0.627200,0.437777,...,0.832382,0.940288,0.251300,0.115464,0.042536,0.622759,0.627993,0.970202,0.310889,0.002891
8988,0.280112,0.664119,0.788992,0.147300,0.330245,0.851793,0.042186,0.915619,0.627159,0.437787,...,0.855944,0.940287,0.251300,0.115467,0.042533,0.585114,0.661190,0.970070,0.247963,0.008310


import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# Calculate correlations with target
correlations = {}
for col in train_processed.columns:
    if col != 'target':
        corr, p_value = pearsonr(train_processed[col], train_processed['target'])
        correlations[col] = {'correlation': corr, 'p_value': p_value}

# Sort by absolute correlation
sorted_correlations = sorted(correlations.items(), 
                            key=lambda x: abs(x[1]['correlation']), 
                            reverse=True)

# Print correlation summary
print("=" * 60)
print("CORRELATION ANALYSIS WITH TARGET")
print("=" * 60)
for col, stats in sorted_correlations:
    corr = stats['correlation']
    p_val = stats['p_value']
    strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
    print(f"{col:30s} | Corr: {corr:7.4f} | p-value: {p_val:.4e} | {strength}")
print("=" * 60)

# Plot each feature with improved visualization
n_samples = 2000
fig_size = (14, 6)

for col, stats in sorted_correlations:
    print(col)
    if True:
        corr = stats['correlation']
        
        fig, axes = plt.subplots(1, 2, figsize=fig_size)
        
        # Left plot: Time series comparison with scatter plots
        ax1 = axes[0]
        ax1_twin = ax1.twinx()
        
        # Create x-axis (sample indices)
        x_indices = np.arange(n_samples)
        
        # Scatter plot for feature
        scatter1 = ax1.scatter(x_indices, train_processed[col][:n_samples], 
                              color='steelblue', s=50, label=col, alpha=0.7, 
                              edgecolors='darkblue', linewidth=0.5)
        
        # Scatter plot for target
        scatter2 = ax1_twin.scatter(x_indices, train_processed['target'][:n_samples], 
                                   color='coral', s=50, label='target', alpha=0.7,
                                   edgecolors='darkred', linewidth=0.5, marker='s')
        
        ax1.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
        ax1.set_ylabel(col, color='steelblue', fontsize=11, fontweight='bold')
        ax1_twin.set_ylabel('Target', color='coral', fontsize=11, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor='steelblue')
        ax1_twin.tick_params(axis='y', labelcolor='coral')
        ax1.grid(True, alpha=0.3)
        
        # Combined legend
        ax1.legend([scatter1, scatter2], [col, 'target'], loc='upper left')
        
        # Right plot: Scatter plot (feature vs target)
        ax2 = axes[1]
        scatter = ax2.scatter(train_processed[col][:n_samples], 
                             train_processed['target'][:n_samples],
                             alpha=0.6, c=range(n_samples), cmap='viridis', 
                             edgecolors='black', linewidth=0.5, s=60)
        
        # Add regression line
        z = np.polyfit(train_processed[col][:n_samples], 
                       train_processed['target'][:n_samples], 1)
        p = np.poly1d(z)
        x_line = np.linspace(train_processed[col][:n_samples].min(), 
                            train_processed[col][:n_samples].max(), 100)
        ax2.plot(x_line, p(x_line), "r--", linewidth=2, label='Linear fit')
        
        ax2.set_xlabel(col, fontsize=11, fontweight='bold')
        ax2.set_ylabel('Target', fontsize=11, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax2)
        cbar.set_label('Sample Index', fontsize=10)
        
        # Overall title
        strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
        fig.suptitle(f'{col} vs Target | Correlation: {corr:.4f} ({strength})', 
                     fontsize=14, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        plt.show()

# Optional: Create a correlation heatmap summary
print("\nTop 10 Most Correlated Features:")
for i, (col, stats) in enumerate(sorted_correlations[:10], 1):
    print(f"{i:2d}. {col:30s} : {stats['correlation']:7.4f}")

In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {'boosting_type': 'gbdt', 
               'objective': 'regression_l1', 
               'metric': 'mape', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 218471017.2888
Training Model 2...
Cumulative Training MAPE: 210993566.8092
Training Model 3...
Cumulative Training MAPE: 205643455.5130
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 205643455.5130
MAE: 0.0001
RMSE: 0.0004


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)


def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match 